# THEMIS 100 m thermal inertia for Gale Crater

Reads the USGS **THEMIS Thermal Inertia Mosaic, Quantitative (32-bit), 100 m/pixel** tile that contains Gale, crops it to the
team bounding box, checks it against the TES (3 km) thermal inertia you already exported, and writes model-ready GeoTIFFs.

**Which file.** The collection is at
[Astropedia: Mars THEMIS-Derived Global Thermal Inertia Mosaic](https://astrogeology.usgs.gov/search/map/mars-themis-derived-global-thermal-inertia-mosaic).
Use the **Quantitative 30S120E** tile (tiles are 30 deg latitude x 60 deg longitude; 30S120E should span 30S-0 and 120E-180E,
so it contains Gale at about 137E, 5S). Do **not** use the "Qualitative" 8-bit tiles, and not the ASU `nightir_100mpix` `.pgm` files:
both are brightness-stretched images, not thermal inertia.
The collection page says tiles are named by the "lower right corner", while the 30S060E record shows 60-120E (lower-left naming).
If the check in step 2 says Gale is outside the tile, or the TES comparison in step 4 shows no correlation, try the `30S180E` tile.

**Format.** The product is distributed as an ISIS3 `.cub` (about 2.5 GB per tile), with GeoTIFF/PNG versions also mentioned.
`rasterio` reads `.cub` through GDAL's ISIS3 driver. If it cannot open it, convert once with
`gdal_translate THEMIS_TI_Mosaic_Quant_30S120E_100mpp.cub tile.tif` and point `TILE_PATH` at the result.

**If the kernel crashes** (exit code 3228369023, hex `0xC06D007F`, often with "GDAL_DATA is not defined"): that code means Windows could not
find a function inside a compiled library. It usually comes from two copies of the GDAL/numpy/PROJ libraries clashing in one environment, for example
a conda environment started without being activated, or `rasterio` installed with both `pip` and `conda`. Restart the kernel, run the
"Environment guard" cell (step 0) first, and if it still crashes run the optional diagnostic cell after step 1. It tests each library in a separate
process and shows which one fails.

**Caveats (from the USGS page).** Apparent thermal inertia from nighttime images: absolute accuracy about 20%, relative differences
within one image about 10%. Image-to-image offsets can reach 50-120 tiu. Slopes, water-ice clouds and shallow ice/bedrock are not modelled,
so crater walls and Mount Sharp flanks are less reliable. Cite Fergason et al. (2006) and the USGS product.

## 0. Environment guard (run first, right after a kernel restart)

Does nothing on Linux/macOS or in a plain `pip` virtual environment. In a conda-style environment on Windows it puts the environment's own
library folders first and tells GDAL/PROJ where their data files are, which is what "GDAL_DATA is not defined" is complaining about.

In [6]:
import os, sys
if os.name == "nt":
    lib = os.path.join(sys.prefix, "Library")
    dirs = [os.path.join(lib, "bin"), os.path.join(lib, "mingw-w64", "bin"), os.path.join(lib, "usr", "bin"),
            os.path.join(sys.prefix, "Scripts")]
    dirs = [d for d in dirs if os.path.isdir(d)]
    if dirs:                                              # conda-style layout found
        os.environ["PATH"] = os.pathsep.join(dirs + [os.environ.get("PATH", "")])
        for d in dirs:
            try:
                os.add_dll_directory(d)
            except OSError:
                pass
        gd, pj = os.path.join(lib, "share", "gdal"), os.path.join(lib, "share", "proj")
        if os.path.isdir(gd):
            os.environ["GDAL_DATA"] = gd
        if os.path.isdir(pj):
            os.environ["PROJ_DATA"] = pj
            os.environ["PROJ_LIB"] = pj
        print("conda-style environment: library folders and GDAL/PROJ data paths set")
    else:
        print("no conda-style folders under", sys.prefix, "- nothing to do")
else:
    print("not Windows - nothing to do")
print("python:", sys.executable)

conda-style environment: library folders and GDAL/PROJ data paths set
python: c:\Users\aypen\anaconda3\envs\GeoSpatialPipeline\python.exe


## 1. Settings

In [19]:
from pathlib import Path
import numpy as np

TILE_PATH = Path("Heatmapdata/THEMIS_TI_Mosaic_Quant_30S120E_100mpp.cub")   # or the .tif version
TILE_LON  = (120.0, 180.0)     # nominal tile extent, degrees East (30S120E)
TILE_LAT  = (-30.0, 0.0)       # nominal tile extent, degrees North

GALE_BBOX = (136.75, -6.57, 139.24, -4.15)   # team bbox: west, south, east, north
PAD_DEG   = 0.0                              # extra margin around the bbox

TES_NIGHT_TIF = Path("processed_albedo_inertia/gale_ti_night_native.tif")   # from the first notebook
TEMPLATE      = Path("Heatmapdata/gale_dem_20m.tif")                          # one of your DEM/slope/aspect tifs

OUT_DIR = Path("processed_albedo_inertia"); OUT_DIR.mkdir(exist_ok=True)

### Optional: environment diagnostic

Set `RUN_DIAG = True` and run the next cell **only if the kernel keeps crashing**. Each library is tested in its own process, so a crash
appears as a FAIL line with its exit code instead of killing the kernel. The first FAIL is the culprit: `numpy`, `scipy` or `matplotlib`
means a numpy/scipy install problem; `rasterio` means GDAL/PROJ. Set it back to `False` afterwards.

In [8]:
RUN_DIAG = False
if RUN_DIAG:
    import subprocess
    print("GDAL_DATA:", os.environ.get("GDAL_DATA"), "| PROJ_DATA:", os.environ.get("PROJ_DATA") or os.environ.get("PROJ_LIB"))
    tes_p, tile_p = repr(str(TES_NIGHT_TIF)), repr(str(TILE_PATH))
    tests = {
        "numpy": "import numpy as np; np.bincount(np.arange(10)); print(np.__version__)",
        "scipy.ndimage": "import numpy as np; from scipy.ndimage import map_coordinates as m; print(m(np.eye(3), [[1.5],[1.5]], order=1))",
        "matplotlib (Agg)": "import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as p, io; p.figure(); p.savefig(io.BytesIO()); print(matplotlib.__version__)",
        "rasterio import": "import rasterio; print(rasterio.__version__, 'GDAL', rasterio.__gdal_version__)",
        "open + read TES tif": f"import rasterio; t = rasterio.open({tes_p}); print(t.read(1).shape, t.crs is not None)",
        "open THEMIS tile": f"import rasterio; s = rasterio.open({tile_p}); print(s.width, s.height)",
        "read THEMIS window": f"import rasterio; from rasterio.windows import Window; s = rasterio.open({tile_p}); print(s.read(1, window=Window(0, 0, 256, 256)).shape)",
    }
    for name, snippet in tests.items():
        p = subprocess.run([sys.executable, "-c", snippet], capture_output=True, text=True, timeout=600)
        rc = p.returncode & 0xFFFFFFFF
        out = (p.stdout.strip().splitlines() or [""])[-1]
        err = (p.stderr.strip().splitlines() or [""])[-1]
        print(f"{name:22s} {'OK  ' if rc == 0 else 'FAIL'} exit {rc:#010x}  {out if rc == 0 else err[:100]}")

## 2. Open the tile and locate Gale

The window is computed from the tile's pixel counts and its nominal 60 x 30 degree extent, so it does not depend on how GDAL
interprets the cube's map projection (the ISIS3 driver often reports metres, not degrees). The location is verified in step 4.
Expect the window to be about 1,475 columns x 1,435 rows at 100 m.

In [11]:
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
from rasterio.crs import CRS

src = rasterio.open(TILE_PATH)
print("size (cols x rows):", src.width, "x", src.height, "| dtype:", src.dtypes[0], "| nodata:", src.nodata)
print("crs:", src.crs)
print("bounds:", src.bounds)

def tile_window(src, bbox, pad=0.0):
    w, s, e, n = bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad
    if not (TILE_LON[0] <= w and e <= TILE_LON[1] and TILE_LAT[0] <= s and n <= TILE_LAT[1]):
        raise ValueError(f"bbox {(w, s, e, n)} is not inside the tile extent lon {TILE_LON}, lat {TILE_LAT}. "
                         "Wrong tile? Try the 30S180E tile (update TILE_LON) or check the tile name.")
    ppd_x = src.width  / (TILE_LON[1] - TILE_LON[0])      # pixels per degree, about 592.8
    ppd_y = src.height / (TILE_LAT[1] - TILE_LAT[0])
    c0 = int(np.floor((w - TILE_LON[0]) * ppd_x)); c1 = int(np.ceil((e - TILE_LON[0]) * ppd_x))
    r0 = int(np.floor((TILE_LAT[1] - n) * ppd_y)); r1 = int(np.ceil((TILE_LAT[1] - s) * ppd_y))
    c0, r0 = max(c0, 0), max(r0, 0); c1, r1 = min(c1, src.width), min(r1, src.height)
    win = Window(c0, r0, c1 - c0, r1 - r0)
    west  = TILE_LON[0] + c0 / ppd_x
    north = TILE_LAT[1] - r0 / ppd_y
    transform = from_origin(west, north, 1.0 / ppd_x, 1.0 / ppd_y)
    return win, transform, (ppd_x, ppd_y)

win, themis_tf, (ppd_x, ppd_y) = tile_window(src, GALE_BBOX, PAD_DEG)
print(f"pixels/deg: x {ppd_x:.2f}, y {ppd_y:.2f}  (expect about 592.8, i.e. 99.7 m/pixel)")
print("window (col_off, row_off, width, height):", win.col_off, win.row_off, win.width, win.height)

size (cols x rows): 35565 x 17782 | dtype: float32 | nodata: -3.4028226550889045e+38
crs: PROJCS["SimpleCylindrical Mars",GEOGCS["GCS_Mars",DATUM["D_Mars",SPHEROID["Mars",3396190,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Equirectangular"],PARAMETER["standard_parallel_1",0],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
bounds: BoundingBox(left=7112900.0, bottom=-1778200.0, right=10669400.0, top=0.0)
pixels/deg: x 592.75, y 592.73  (expect about 592.8, i.e. 99.7 m/pixel)
window (col_off, row_off, width, height): 9928 2459 1477 1436


## 3. Read the window and clean it

ISIS marks missing pixels with a huge negative number (about -3.4e38). Those, NaN and values <= 0 become NaN.
Nothing else is changed, because USGS says the quantitative values are unmodified by the mosaicking.

In [12]:
def read_ti(src, window):
    arr = src.read(1, window=window).astype("float32")
    bad = ~np.isfinite(arr) | (arr < -1e30) | (arr <= 0)
    if src.nodata is not None and np.isfinite(src.nodata):
        bad |= (arr == np.float32(src.nodata))
    arr[bad] = np.nan
    return arr

themis = read_ti(src, win)
ok = np.isfinite(themis)
print("shape:", themis.shape, f"| valid: {ok.mean():.1%}")
print("percentiles 1/25/50/75/99 (tiu):", np.round(np.nanpercentile(themis, [1, 25, 50, 75, 99]), 0))
print("min / max:", np.nanmin(themis), np.nanmax(themis))

shape: (1436, 1477) | valid: 100.0%
percentiles 1/25/50/75/99 (tiu): [171. 234. 281. 336. 541.]
min / max: 102.23149 818.64307


C:\Users\aypen\AppData\Local\Temp\ipykernel_1092\3455235572.py:2: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1, window=window).astype("float32")


## 4. Check against your TES thermal inertia (location and units)

THEMIS pixels are averaged into the 0.05 degree TES pixels and compared. A wrong tile or window usually shows up as weak correlation,
a ratio far from 1, or both; a wrong unit shows up as a ratio far from 1 (correlation alone can look fine). Differences of 20-30% are normal
(different instruments, models and dates). Also look at the preview plot in step 7: the large-scale patterns should match.

In [13]:
with rasterio.open(TES_NIGHT_TIF) as t:
    tes = t.read(1).astype("float64"); tes_tf = t.transform
nr, nc = tes.shape

lon = themis_tf.c + (np.arange(themis.shape[1]) + 0.5) * themis_tf.a
lat = themis_tf.f + (np.arange(themis.shape[0]) + 0.5) * themis_tf.e     # e is negative
cols = np.floor((lon - tes_tf.c) / tes_tf.a).astype(int)
rows = np.floor((lat - tes_tf.f) / tes_tf.e).astype(int)
inside = (rows[:, None] >= 0) & (rows[:, None] < nr) & (cols[None, :] >= 0) & (cols[None, :] < nc) & ok
idx = (rows[:, None] * nc + cols[None, :])[inside]
sums = np.bincount(idx, weights=themis[inside], minlength=nr * nc).reshape(nr, nc)
cnts = np.bincount(idx, minlength=nr * nc).reshape(nr, nc)
full = cnts >= 0.6 * cnts.max()                       # keep TES pixels well covered by THEMIS
themis_on_tes = np.where(full, sums / np.maximum(cnts, 1), np.nan)

both = full & np.isfinite(tes)
ratio = themis_on_tes[both] / tes[both]
r = np.corrcoef(themis_on_tes[both], tes[both])[0, 1]
RATIO_MED = float(np.median(ratio))
print(f"TES pixels compared: {both.sum()}")
print(f"correlation r = {r:.2f}")
print(f"THEMIS / TES ratio: median {RATIO_MED:.2f}, 25-75% {np.percentile(ratio, 25):.2f}-{np.percentile(ratio, 75):.2f}")
if both.sum() < 100 or r < 0.5:
    print("WARNING: weak agreement. Wrong tile, wrong window, or different data. Do not use this output yet.")
elif not 0.6 <= RATIO_MED <= 1.6:
    print("WARNING: ratio far from 1. Check the units of the tile before using it.")
else:
    print("OK: location and units look consistent with TES.")

TES pixels compared: 2400
correlation r = 0.43
THEMIS / TES ratio: median 1.18, 25-75% 1.01-1.39


C:\Users\aypen\AppData\Local\Temp\ipykernel_1092\1033023275.py:2: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  tes = t.read(1).astype("float64"); tes_tf = t.transform


## 5. Write GeoTIFFs at 100 m

`themis100m` is THEMIS only. `merged100m` fills THEMIS gaps with the TES map (sampled smoothly and scaled by the median ratio above),
and `source_flag` marks which pixels came from where (1 = THEMIS, 0 = TES). Use `themis100m` if you prefer to keep gaps as NaN.

In [15]:
from scipy.ndimage import map_coordinates

MARS_CRS = CRS.from_proj4("+proj=longlat +a=3396190 +b=3396190 +no_defs")

def write_tif(path, arr, transform, crs=MARS_CRS):
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype="float32", crs=crs, transform=transform,
                       nodata=np.nan, compress="deflate") as dst:
        dst.write(arr.astype("float32"), 1)

# TES sampled bilinearly at every THEMIS pixel centre
rr = ((lat - tes_tf.f) / tes_tf.e - 0.5)[:, None] * np.ones((1, len(lon)))
cc = ((lon - tes_tf.c) / tes_tf.a - 0.5)[None, :] * np.ones((len(lat), 1))
tes_on_themis = map_coordinates(tes, [rr, cc], order=1, mode="nearest").astype("float32") * np.float32(RATIO_MED)

merged = np.where(ok, themis, tes_on_themis).astype("float32")
source_flag = ok.astype("float32")

write_tif(OUT_DIR / "gale_ti_night_themis100m_native.tif", themis, themis_tf)
write_tif(OUT_DIR / "gale_ti_night_merged100m_native.tif", merged, themis_tf)
write_tif(OUT_DIR / "gale_ti_night_source_flag_native.tif", source_flag, themis_tf)
print("written to", OUT_DIR, ":\n  gale_ti_night_themis100m_native.tif\n  gale_ti_night_merged100m_native.tif\n  gale_ti_night_source_flag_native.tif")
print(f"THEMIS pixels: {ok.mean():.1%}   filled from TES: {(~ok).mean():.1%}")

written to processed_albedo_inertia :
  gale_ti_night_themis100m_native.tif
  gale_ti_night_merged100m_native.tif
  gale_ti_night_source_flag_native.tif
THEMIS pixels: 100.0%   filled from TES: 0.0%


## 6. Match your model grid (200 m DEM, slope, aspect)

Each layer is area-averaged (100 m to 200 m) onto exactly the grid of `TEMPLATE`, so everything lines up pixel for pixel.
Check the output against the DEM before trusting it: the two files may define the Mars sphere slightly differently.

In [20]:
from rasterio.warp import reproject, Resampling

def to_template(arr, src_transform, template_path, method=Resampling.average):
    with rasterio.open(template_path) as t:
        dst = np.full((t.height, t.width), np.nan, dtype=np.float32)
        reproject(source=arr.astype("float32"), destination=dst,
                  src_transform=src_transform, src_crs=MARS_CRS,
                  dst_transform=t.transform, dst_crs=t.crs,
                  src_nodata=np.nan, dst_nodata=np.nan, resampling=method)
        return dst, t.transform, t.crs

if TEMPLATE.exists():
    for name, arr in (("themis", themis), ("merged", merged)):
        g, tf, crs = to_template(arr, themis_tf, TEMPLATE)
        write_tif(OUT_DIR / f"gale_ti_night_{name}_grid.tif", g, tf, crs)
        print(name, "on template grid:", g.shape, f"valid {np.isfinite(g).mean():.1%}")
else:
    print(f"{TEMPLATE} not found; skipped. Set TEMPLATE to one of your DEM/slope/aspect .tif files.")

themis on template grid: (7168, 7680) valid 93.1%
merged on template grid: (7168, 7680) valid 93.1%


## 7. Optional preview

Saved to a PNG (matplotlib imported only here, so a broken install cannot affect anything above). Set `PLOT = False` to skip.

In [14]:
PLOT = True
if PLOT:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ext = [themis_tf.c, themis_tf.c + themis.shape[1] * themis_tf.a,
               themis_tf.f + themis.shape[0] * themis_tf.e, themis_tf.f]
        fig, ax = plt.subplots(1, 2, figsize=(13, 5))
        vmin, vmax = np.nanpercentile(themis, [2, 98])
        im = ax[0].imshow(themis, extent=ext, cmap="viridis", vmin=vmin, vmax=vmax)
        ax[0].set_title("THEMIS night thermal inertia, 100 m (tiu)"); fig.colorbar(im, ax=ax[0])
        im = ax[1].imshow(tes, extent=[tes_tf.c, tes_tf.c + nc * tes_tf.a, tes_tf.f + nr * tes_tf.e, tes_tf.f],
                          cmap="viridis", vmin=vmin, vmax=vmax)
        ax[1].set_title("TES night thermal inertia, 3 km (tiu)"); fig.colorbar(im, ax=ax[1])
        for a in ax:
            a.set_xlabel("East longitude"); a.set_ylabel("Latitude")
        fig.tight_layout(); fig.savefig(OUT_DIR / "themis_vs_tes.png", dpi=100)
        print("saved", OUT_DIR / "themis_vs_tes.png")
    except Exception as ex:
        print("plot skipped:", ex)

saved processed_albedo_inertia\themis_vs_tes.png
